# Dataset loading

This section loads the local Polish road sign classification dataset and checks the number of images in each class.

In [ ]:
from pathlib import Path
import math
import random
import shutil

import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
DATA_DIR = Path("../data")
CLASSIFICATION_DIR = DATA_DIR / "classification"

CLASSIFICATION_DIR.exists()


In [ ]:
class_dirs = sorted([path for path in CLASSIFICATION_DIR.iterdir() if path.is_dir()])
class_names = [path.name for path in class_dirs]

class_names


In [ ]:
image_extensions = {".jpg", ".jpeg", ".png"}

class_counts = []

for class_dir in class_dirs:
    image_count = len([
        path for path in class_dir.rglob("*")
        if path.suffix.lower() in image_extensions
    ])

    class_counts.append({
        "class_name": class_dir.name,
        "image_count": image_count,
    })

df_classes = pd.DataFrame(class_counts)
df_classes


In [ ]:
total_images = df_classes["image_count"].sum()
total_images


# Exploratory data analysis

This section checks the basic structure of the classification dataset.

In [ ]:
num_classes = len(df_classes)
num_classes


In [ ]:
df_classes.sort_values("image_count").head()


In [ ]:
df_classes.sort_values("image_count", ascending=False).head()


In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(df_classes["class_name"], df_classes["image_count"])
plt.xticks(rotation=90)
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.title("Number of images per class")
plt.show()


# Sample road sign images

This section displays example images from selected road sign classes.

In [ ]:
sample_images = []

for class_dir in class_dirs:
    image_paths = sorted([
        path for path in class_dir.rglob("*")
        if path.suffix.lower() in image_extensions
    ])

    if image_paths:
        sample_images.append({
            "class_name": class_dir.name,
            "image_path": image_paths[0],
        })

len(sample_images)


In [ ]:
columns = 4
rows = math.ceil(len(sample_images) / columns)

fig, axes = plt.subplots(rows, columns, figsize=(12, rows * 3))
axes = axes.flatten()

for ax, sample in zip(axes, sample_images):
    image = plt.imread(sample["image_path"])
    ax.imshow(image)
    ax.set_title(sample["class_name"])
    ax.axis("off")

for ax in axes[len(sample_images):]:
    ax.axis("off")

plt.tight_layout()
plt.show()


# Data preprocessing

This section prepares the classification dataset split for YOLO11 training.

In [ ]:
PROCESSED_DATA_DIR = DATA_DIR / "processed" / "yolo11_classification"

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42

TRAIN_RATIO + VAL_RATIO + TEST_RATIO


In [ ]:
image_records = []

for class_dir in class_dirs:
    image_paths = sorted([
        path for path in class_dir.rglob("*")
        if path.suffix.lower() in image_extensions
    ])

    for image_path in image_paths:
        image_records.append({
            "class_name": class_dir.name,
            "image_path": image_path,
        })

df_images = pd.DataFrame(image_records)
df_images.head()


In [ ]:
rng = random.Random(RANDOM_SEED)
split_records = []

for class_name, group in df_images.groupby("class_name"):
    image_paths = group["image_path"].tolist()
    rng.shuffle(image_paths)

    total_count = len(image_paths)
    train_count = int(total_count * TRAIN_RATIO)
    val_count = int(total_count * VAL_RATIO)

    train_paths = image_paths[:train_count]
    val_paths = image_paths[train_count:train_count + val_count]
    test_paths = image_paths[train_count + val_count:]

    for split_name, paths in [
        ("train", train_paths),
        ("val", val_paths),
        ("test", test_paths),
    ]:
        for image_path in paths:
            split_records.append({
                "split": split_name,
                "class_name": class_name,
                "image_path": image_path,
            })

df_split = pd.DataFrame(split_records)
df_split.head()


In [ ]:
split_summary = (
    df_split
    .groupby(["split", "class_name"])
    .size()
    .reset_index(name="image_count")
)

split_summary.head()


In [ ]:
df_split["split"].value_counts()


In [ ]:
def prepare_yolo11_classification_dataset(df_split, output_dir):
    output_dir = Path(output_dir)

    for row in df_split.itertuples(index=False):
        destination_dir = output_dir / row.split / row.class_name
        destination_dir.mkdir(parents=True, exist_ok=True)

        destination_path = destination_dir / row.image_path.name
        shutil.copy2(row.image_path, destination_path)

    return output_dir


In [ ]:
COPY_DATASET = False

if COPY_DATASET:
    prepare_yolo11_classification_dataset(df_split, PROCESSED_DATA_DIR)

PROCESSED_DATA_DIR
